In [1]:
import csv
import tiktoken
import pandas as pd
import os
from tqdm.auto import tqdm
import json

In [2]:
all_data_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_relevance_symbols_data.csv')

In [3]:
all_data_df[:3]

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,...,activation_settings,actors,recepients,symbolic_prompt,relevance_judgment,relevance_justification,relevance_metrics,symbolic_annotation,symbolic_metrics,symbolic_quality
0,0,0,0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],...,family,child,parent,Norm ID: 0\nTheme ID: 16\n\nConversation from ...,relevant,Justification: The conversation reflects a str...,"{'clarity': '4 - clear', 'contextuality': '5 -...",Social Norm - Norm Concept Compatibility: does...,{'contextual influence evaluation': 'high infl...,False
1,1,1,1,15,2. Filial piety,0.2567,False,False,[],[],...,family,child,parent,Norm ID: 15\nTheme ID: 16\n\nConversation from...,relevant,Justification: The conversation reflects a str...,"{'clarity': '4 - clear', 'contextuality': '4 -...",Social Norm - Norm Concept Compatibility: matc...,{'contextual influence evaluation': 'high infl...,False
2,2,2,2,18,2. Filial piety: Children are expected to prio...,0.0858,False,False,[],[],...,family,child,parent,Norm ID: 18\nTheme ID: 16\n\nConversation from...,relevant,Justification: The social norm of filial piety...,"{'clarity': '4 - clear', 'contextuality': '5 -...",Social Norm - Norm Concept Compatibility: does...,{'contextual influence evaluation': 'high infl...,False


In [4]:
fi_df = all_data_df[all_data_df['identifier'].str.contains('ldc-')]

In [5]:
print(fi_df.dialogue_id.iloc[2], fi_df.summary.iloc[2], fi_df.identifier.iloc[2])

16105 The conversation revolves around the speaker's commitment to taking care of their elderly parents and their feelings of reluctance to leave them. They reflect on the importance of being a filial child, and acknowledge their own shortcomings in managing their temper. Additionally, they discuss future plans for visits and express anticipation for a possible visit to Beijing. ldc-M01000GE7-12


In [6]:
fi_d_df = fi_df[fi_df.dialogue_id == 16105]

In [7]:
for index, row in fi_d_df.iterrows():
    print(row['text'])
    print(row['symbolic_prompt'])
    print(row['symbolic_annotation'])

1. Filial Piety: The concept of filial piety is deeply ingrained in Chinese culture, emphasizing the respect and care for one's parents, especially as they age. The speaker's commitment to looking after their elderly parents demonstrates adherence to this cultural norm.
Norm ID: 43231
Theme ID: 16

Conversation from Chinese culture:
(neutral): 132624: 不过父母年纪大，以后每年我会接他们来住住
(neutral): 132624: 离开了，心里还真有些不舍
(neutral): 133485: 是啊
(neutral): 133485: 你是孝子
(neutral): 132624: 最基本的
(neutral): 132624: 相处时我不耐烦的时候多
(neutral): 132624: 素质不高
(neutral): 133485: 哈哈
(neutral): 133485: 离开了舍不得了吧
(neutral): 132624: 呵呵
(neutral): 132624: 你什么时候来北京
(neutral): 133485: 不知道
(neutral): 133485: 领导同事照顾我家里孩子小
(neutral): 133485: 一般不安排我出差
(neutral): 132624: 那下次安排了，你要踊跃点

Social Norm:
1. Filial Piety: The concept of filial piety is deeply ingrained in Chinese culture, emphasizing the respect and care for one's parents, especially as they age. The speaker's commitment to looking after their elderly parents demonstrates a

In [8]:
# Convert phase 2 annotations to standard format
ann2 = pd.read_csv('../additional_annotations_norms_v5.csv')
ann2.rename(columns={'display_string': 'dialogue', 'theme_name': 'theme', 'norm_text': 'norm', 'applicable?': 'applicability', 'matches?': 'compatibility', 'violation?': 'violation_status'}, inplace=True)
ann2['dialogue'] = ann2['dialogue'].apply(lambda x:str(x).replace('<b>', '').replace('</b>', '').replace('<br>', '\n').strip())
ann2_filtered = ann2[['dialogue_id', 'dialogue', 'theme', 'norm_id', 'norm', 'summary', 'compatibility', 'applicability', 'violation_status']]
ann2_filtered[:3]
# print(len(ann2), len(ann2_filtered), len(ann2_filtered_dialogues))

,dialogue_id,dialogue,theme,norm_id,norm,summary,compatibility,applicability,violation_status
0,3,nan,KMeans_5,8,1. Respect for elders: Zuo Zhengpeng addresses...,NaN,NaN,TRUE,False
1,3,"Conversation Summary:\nIn this conversation, Z...",HospitalityTowardsGuests,9,2. Hospitality: Mrs. Xu warmly welcomes Zuo Zh...,"In this conversation, Zuo Zhengpeng visits Lih...",True,TRUE,False
2,3,nan,HospitalityTowardsGuests,10,3. Filial piety: Mrs. Xu instructs her husband...,NaN,True,TRUE,False


In [10]:
ann2_dialogues = ann2_filtered[['dialogue_id', 'dialogue', 'summary']]
ann2_dialogues = ann2_dialogues[~ann2_dialogues.dialogue.str.strip().isin(['nan'])]

ann2_filtered_dialogues = pd.merge(ann2_filtered, ann2_dialogues, on='dialogue_id', how='inner')
ann2_filtered_dialogues = ann2_filtered_dialogues[['dialogue_id', 'dialogue_y', 'theme', 'norm_id', 'norm', 'summary_y', 'compatibility', 'applicability', 'violation_status']]
ann2_filtered_dialogues.rename(columns={'dialogue_y': 'dialogue', 'summary_y': 'summary'}, inplace=True)
ann2_filtered_dialogues[:3]

,dialogue_id,dialogue,theme,norm_id,norm,summary,compatibility,applicability,violation_status
0,3,"Conversation Summary:\nIn this conversation, Z...",KMeans_5,8,1. Respect for elders: Zuo Zhengpeng addresses...,"In this conversation, Zuo Zhengpeng visits Lih...",NaN,TRUE,False
1,3,"Conversation Summary:\nIn this conversation, Z...",HospitalityTowardsGuests,9,2. Hospitality: Mrs. Xu warmly welcomes Zuo Zh...,"In this conversation, Zuo Zhengpeng visits Lih...",True,TRUE,False
2,3,"Conversation Summary:\nIn this conversation, Z...",HospitalityTowardsGuests,10,3. Filial piety: Mrs. Xu instructs her husband...,"In this conversation, Zuo Zhengpeng visits Lih...",True,TRUE,False


In [11]:
print(len(ann2_filtered_dialogues.dialogue_id.value_counts()))

532


In [12]:
ann2_filtered_dialogues = ann2_filtered_dialogues[ann2_filtered_dialogues.applicability.str.upper().isin(['TRUE', 'FALSE'])]
ann2_filtered_dialogues['applicability'] = ann2_filtered_dialogues['applicability'].apply(lambda x:x=='TRUE')
ann2_filtered_dialogues['compatibility'] = ann2_filtered_dialogues['compatibility'].apply(lambda x:x==True)
ann2_filtered_dialogues[:3]

,dialogue_id,dialogue,theme,norm_id,norm,summary,compatibility,applicability,violation_status
0,3,"Conversation Summary:\nIn this conversation, Z...",KMeans_5,8,1. Respect for elders: Zuo Zhengpeng addresses...,"In this conversation, Zuo Zhengpeng visits Lih...",False,True,False
1,3,"Conversation Summary:\nIn this conversation, Z...",HospitalityTowardsGuests,9,2. Hospitality: Mrs. Xu warmly welcomes Zuo Zh...,"In this conversation, Zuo Zhengpeng visits Lih...",True,True,False
2,3,"Conversation Summary:\nIn this conversation, Z...",HospitalityTowardsGuests,10,3. Filial piety: Mrs. Xu instructs her husband...,"In this conversation, Zuo Zhengpeng visits Lih...",True,True,False


In [13]:
print(len(ann2_filtered_dialogues))
ann2_comp = ann2_filtered_dialogues[~ann2_filtered_dialogues.theme.str.contains('KMeans')]
print(len(ann2_comp))

726
580


In [30]:
ann2_comp.compatibility.value_counts()

compatibility
True     528
False     52
Name: count, dtype: int64

In [14]:
symbolic_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_annotation_verification_outputs/'
bnames = os.listdir(symbolic_output_dir)
stage1_outs = {}
for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(symbolic_output_dir, bname)))
    for out in bres:
        n_id, _, ann, quality = out
        stage1_outs[n_id] = (ann.strip(), quality.strip()) 
            
print(len(stage1_outs))

  0%|          | 0/41 [00:00<?, ?it/s]

40816


In [15]:
agenteval_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_agenteval_outputs/'
bnames = os.listdir(agenteval_output_dir)
stage2_outs = {}
for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(agenteval_output_dir, bname)))
    for out in bres:
        n_id, _, metrics, ann = out
        stage2_outs[n_id] = (ann.strip(), metrics.strip()) 
       
print(len(stage2_outs))

  0%|          | 0/41 [00:00<?, ?it/s]

40816


In [24]:
#for stage 2 agenteval verification

w, t = 0, 0
cset, rset, vset = {}, {}, {}

stage2_anns = {}
for n_id in tqdm(stage2_outs):
    
    rlines = [l.strip() for l in stage2_outs[n_id][0].split('\n') if l.strip()]
    relevance, compatibility, vio_status = None, None, None
    
    for rline in rlines:
        if rline.startswith('Relevance:'):
            relevance = rline.split('Relevance:')[1].strip()
        if 'Compatibility:' in rline:
            compatibility = rline.split('Compatibility:')[1].strip()
        if rline.startswith('Violation Status:'):
            vio_status = rline.split('Violation Status:')[1].strip()
    if (not relevance) or (not compatibility):
        w += 1
    elif relevance == 'relevant' and (not vio_status):
        w += 1
    else:
        if compatibility not in cset:
            cset[compatibility] = 1
        else:
            cset[compatibility] += 1
        if relevance not in rset:
            rset[relevance] = 1
        else:
            rset[relevance] += 1
        if vio_status not in vset:
            vset[vio_status] = 1
        else:
            vset[vio_status] += 1
    t += 1
    
    stage2_anns[n_id] = (compatibility, relevance, vio_status)
print(w, t)

  0%|          | 0/40816 [00:00<?, ?it/s]

0 40816


In [25]:
print(cset)
print(rset)
print(vset)

{'match': 33009, "doesn't match": 7807}
{'relevant': 38468, 'irrelevant': 2348}
{'violate': 10581, 'adhere': 30235}


In [18]:
#for stage 1 verification

w, t = 0, 0
cset, rset, vset, qset = set(), set(), set(), set()
stage1_anns = {}

for n_id in tqdm(stage1_outs):
    rlines = [l.strip() for l in stage1_outs[n_id][0].split('\n') + stage1_outs[n_id][1].split('\n') if l.strip()]
    relevance, compatibility, vio_status, quality = None, None, None, None
    for rline in rlines:
        if rline.startswith('Relevance:'):
            relevance = rline.split('Relevance:')[1].strip()
        if rline.startswith('Quality Judgment:'):
            quality = rline.split('Quality Judgment:')[1].strip()
        if 'Compatibility:' in rline:
            compatibility = rline.split('Compatibility:')[1].strip()
        if rline.startswith('Violation Status:'):
            vio_status = rline.split('Violation Status:')[1].strip()
    if (not relevance) or (not compatibility) or (not quality):
        w += 1
    elif relevance == 'relevant' and (not vio_status):
        w += 1
    else:
        cset.add(compatibility)
        rset.add(relevance)
        vset.add(vio_status)
        qset.add(quality)
    t += 1
    
    stage1_anns[n_id] = (compatibility, relevance, vio_status, quality)
print(w, t)

  0%|          | 0/40816 [00:00<?, ?it/s]

0 40816


In [19]:
print(cset)
print(rset)
print(vset)
print(qset)

{"doesn't match", 'match'}
{'irrelevant', 'relevant'}
{'irrelevant', 'adhere', 'violate'}
{'inaccurate', 'accurate'}


In [20]:
w = 0
ann2_comp = ann2_filtered_dialogues[~ann2_filtered_dialogues.theme.str.contains('KMeans')]
for index, row in ann2_comp.iterrows():
    n_id = row['norm_id']
    if n_id not in stage1_anns or n_id not in stage2_anns:
        w += 1
print(w)

0


In [59]:
# vc = ann2_filtered_dialogues.applicability.value_counts()
# applicability_stage0 = [vc[True], (vc[True] + vc[False]), vc[True] / (vc[True] + vc[False])]

ann2_comp = ann2_filtered_dialogues[~ann2_filtered_dialogues.theme.str.contains('KMeans')]
vc = ann2_comp.compatibility.value_counts()
compatibility_stage0 = [vc[True], (vc[True] + vc[False]), vc[True] / (vc[True] + vc[False])]

applicability_stage0 = [0, 0, 0]
# applicability_stage2 = [0, 0, 0]

compatibility_stage1 = [0, 0, 0]
compatibility_stage2 = [0, 0, 0]

vstatus_stage0 = [0, 0, 0]
vstatus_stage1 = [0, 0, 0]
vstatus_stage2 = [0, 0, 0]

for index, row in ann2_filtered_dialogues.iterrows():
    n_id = row['norm_id']
    cg = row['compatibility']
    rg = row['applicability']
    vg = row['violation_status']

    if n_id in stage1_anns:
        c1, r1, v1, q1 = stage1_anns[n_id]
        c2, r2, v2 = stage2_anns[n_id]

        if (True):
            if True:#r1 == 'relevant':
                if r1 == 'relevant':
                    applicability_stage0[1] += 1
                if r1 == 'relevant' and rg == True:
                    applicability_stage0[0] += 1
                    
                vstatus_stage0[1] += 1
                if (v1 == 'violate' and vg == True) or (v1 == 'adhere' and vg == False):
                    vstatus_stage0[0] += 1
                if q1 == 'accurate':
                    vstatus_stage1[1] += 1
                if (q1 == 'accurate' and v1 == 'violate' and vg == True) or (q1 == 'accurate' and v1 == 'adhere' and vg == False):
                    vstatus_stage1[0] += 1

                if q1 == 'inaccurate' or (q1 == 'accurate' and c1 == "match"):
                    compatibility_stage1[1] += 1
                if (q1 == 'inaccurate' and cg == True) or (q1 == 'accurate' and cg == True and c1 == "match"):
                    compatibility_stage1[0] += 1
    
            if c2 == 'match' and r2 == 'relevant':
                # applicability_stage2[1] += 1
                vstatus_stage2[1] += 1
                if (v2 == 'violate' and vg == True) or (v2 == 'adhere' and vg == False):
                    vstatus_stage2[0] += 1
    
                if c2 == 'match':
                    compatibility_stage2[1] += 1
                if (c2 == 'match' and cg == True):
                    compatibility_stage2[0] += 1

print('Stage 0:')
print(f"Compatibility: {round(100 * compatibility_stage0[0] / compatibility_stage0[1], 2)}")
print(f"Applicability: {round(100 * applicability_stage0[0] / applicability_stage0[1], 2)}, Retention: {round(100 * applicability_stage0[1] / 726, 2)}, Corr-Retention: {round(100 * applicability_stage0[0] / 588, 2)}")
print(f"Violation Status: {round(100 * vstatus_stage0[0] / vstatus_stage0[1], 2)}\n")

print('Stage 1:')
print(f"Compatibility: {round(100 * compatibility_stage1[0] / compatibility_stage1[1], 2)}, Retention: {round(100 * compatibility_stage1[1] / 580, 2)}, Corr-Retention: {round(100 * compatibility_stage1[0] / 528, 2)}")
# print(f"Applicability: {round(100 * applicability_stage1[0] / applicability_stage1[1], 2)}")
print(f"Violation Status: {round(100 * vstatus_stage1[0] / vstatus_stage1[1], 2)}, Retention: {round(100 * vstatus_stage1[1] / 580, 2)}, Corr-Retention: {round(100 * vstatus_stage1[0] / vstatus_stage0[0], 2)}\n")

print('Stage 2:')
print(f"Compatibility: {round(100 * compatibility_stage2[0] / compatibility_stage2[1], 2)}, Retention: {round(100 * compatibility_stage2[1] / 580, 2)}, Corr-Retention: {round(100 * compatibility_stage2[0] / 528, 2)}")
# print(f"Applicability: {round(100 * applicability_stage2[0] / applicability_stage2[1], 2)}")
print(f"Violation Status: {round(100 * vstatus_stage2[0] / vstatus_stage2[1], 2)}, Retention: {round(100 * vstatus_stage2[1] / 580, 2)}, Corr-Retention: {round(100 * vstatus_stage2[0] / vstatus_stage0[0], 2)}")

Stage 0:
Compatibility: 91.03
Applicability: 82.18, Retention: 71.9, Corr-Retention: 72.96
Violation Status: 60.34

Stage 1:
Compatibility: 93.4, Retention: 91.38, Corr-Retention: 93.75
Violation Status: 64.27, Retention: 69.48, Corr-Retention: 74.0

Stage 2:
Compatibility: 93.58, Retention: 83.28, Corr-Retention: 85.61
Violation Status: 59.01, Retention: 83.28, Corr-Retention: 81.43


In [29]:
# applicability is relevant to all norm descriptions
# hence can't be validated with symbolic annotation

# vc = ann2_filtered_dialogues.applicability.value_counts()
# applicability_stage0 = [vc[True], (vc[True] + vc[False]), vc[True] / (vc[True] + vc[False])]

# compatibility on the other hand is relevant only for named norm concepts
# hence, can be measured in symbolic annotation

ann2_comp = ann2_filtered_dialogues[~ann2_filtered_dialogues.theme.str.contains('KMeans')]
vc = ann2_comp.compatibility.value_counts()
compatibility_stage0 = [vc[True], (vc[True] + vc[False]), vc[True] / (vc[True] + vc[False])]

# applicability_stage1 = [0, 0, 0]
# applicability_stage2 = [0, 0, 0]

compatibility_stage1 = [0, 0, 0]
compatibility_stage2 = [0, 0, 0]

# while violation status is applicable for all relevant norms, we only got symbolic information 
# for named norm concepts

vstatus_stage1 = [0, 0, 0]
vstatus_stage2 = [0, 0, 0]

for index, row in ann2_filtered_dialogues.iterrows():
    n_id = row['norm_id']
    cg = row['compatibility']
    # rg = row['applicability']
    vg = row['violation_status']

    if n_id in stage1_anns:
        c1, r1, v1, q1 = stage1_anns[n_id]
        c2, r2, v2 = stage2_anns[n_id]

        if True: #q1 == 'accurate':
            if r1 == 'relevant':
                # applicability_stage1[1] += 1
                vstatus_stage1[1] += 1
                # if rg == True:
                #     applicability_stage1[0] += 1
                if (v1 == 'violate' and vg == True) or (v1 == 'adhere' and vg == False):
                    vstatus_stage1[0] += 1
    
            if (c1 == 'match' and cg == True):
                compatibility_stage1[0] += 1
            if c1 == 'match':
                compatibility_stage1[1] += 1
    
            if r2 == 'relevant':
                # applicability_stage2[1] += 1
                vstatus_stage2[1] += 1
                # if rg == True:
                #     applicability_stage2[0] += 1
                if (v2 == 'violate' and vg == True) or (v2 == 'adhere' and vg == False):
                    vstatus_stage2[0] += 1
    
            if (c2 == 'match' and cg == True):
                compatibility_stage2[0] += 1
            if c2 == 'match':
                compatibility_stage2[1] += 1

print('Stage 0:')
print(f"Compatibility: {round(100 * compatibility_stage0[0] / compatibility_stage0[1], 2)}, {round(100 * compatibility_stage0[1] / 580, 2)}\n")
# print(f"Applicability: {round(100 * applicability_stage0[0] / applicability_stage0[1], 2)}, {round(100 * applicability_stage0[1] / 726, 2)}\n")

print('Stage 1:')
print(compatibility_stage1)
print(f"Compatibility: {round(100 * compatibility_stage1[0] / compatibility_stage1[1], 2)}, {round(100 * compatibility_stage1[1] / 580, 2)}")
# print(f"Applicability: {round(100 * applicability_stage1[0] / applicability_stage1[1], 2)}, {round(100 * applicability_stage1[1] / 726, 2)}")
print(f"Violation Status: {round(100 * vstatus_stage1[0] / vstatus_stage1[1], 2)}, {round(100 * vstatus_stage1[1] / 580, 2)}\n")

print('Stage 2:')
print(f"Compatibility: {round(100 * compatibility_stage2[0] / compatibility_stage2[1], 2)}, {round(100 * compatibility_stage2[1] / 580, 2)}")
# print(f"Applicability: {round(100 * applicability_stage2[0] / applicability_stage2[1], 2)}, {round(100 * applicability_stage2[1] / 726, 2)}")
print(f"Violation Status: {round(100 * vstatus_stage2[0] / vstatus_stage2[1], 2)}, {round(100 * vstatus_stage2[1] / 580, 2)}")

Stage 0:
Compatibility: 91.03, 100.0

Stage 1:
[476, 502, 0]
Compatibility: 94.82, 86.55
Violation Status: 61.88, 90.0

Stage 2:
Compatibility: 93.58, 83.28
Violation Status: 60.64, 96.38


In [ ]:
tp, t = 0, 0
tpq, tq = 0, 0
for index, row in ann2_filtered_dialogues.iterrows():
    n_id = row['norm_id']
    if n_id in sym_anns:
        cg, rg, vg = row['compatibility'], row['applicability'], row['violation_status']
        c, r, v, q = sym_anns[n_id]
        if vg == False and v == 'violate' and q == 'accurate':
            # or ((vg == False and (v == 'violate' or v == 'irrelevant')) or (vg == True and (v == 'adhere' or v == 'irrelevant')))) or (rg == True and r == 'relevant'))  (rg == False and r == 'irrelevant')
            print(n_id)
            print(cg, rg, vg)
            print(c, r, v, q)
            break

In [74]:
all_data_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_relevance_data.csv')

In [75]:
print(all_data_df.columns)

Index(['Unnamed: 0.1', 'Unnamed: 0', 'id__x', 'text', 'distance', 'good',
       'validated', 'gpt3_answer_rating', 'gpt3_feedback', 'actor_role_id',
       'recepient_role_id', 'dialogue_id', 'theme_id', 'id__y', 'identifier',
       'culture', 'dialogue_string', 'display_text', 'summary',
       'other_features', 'dialogue_text_zh', 'id_', 'name', 'description',
       'violation_characteristic', 'activation_settings', 'actors',
       'recepients', 'symbolic_prompt', 'relevance_judgment',
       'relevance_justification', 'relevance_metrics'],
      dtype='object')


In [76]:
symbols = {}
metrics = {}
quality = {}

for n_id in stage1_outs:
    c, r, v, q = stage1_anns[n_id]
    # print(q)
    # break
    ann1, qual = stage1_outs[n_id]
    symbols[n_id] = f"{ann1.strip()}\n{qual.strip()}"
    quality[n_id] = True if q.strip().lower() == 'accurate' else False
    ann2, m = stage2_outs[n_id]
    if m.startswith('```json'):
        m = m[7:-3]
    mets = json.loads(m.strip())
    metrics[n_id] = mets

In [77]:
t, f = 0, 0
for n_id in quality:
    c, r, v, q = stage1_anns[n_id]
    if quality[n_id]:
        t += 1
    else:
        f += 1
print(t, f)

25702 15114


In [78]:
for n_id in quality:
    print(quality[n_id])
    print(symbols[n_id])
    print(metrics[n_id])
    break

False
Social Norm - Norm Concept Compatibility: match  
Compatibility Justification: The social norm emphasizes privacy and trust in relationships, which aligns with the expectation of confidentiality regarding personal matters shared within a close circle.

Relevance: relevant  
Relevance Justification: The conversation discusses marital issues and the sharing of personal information, making the norm highly relevant to the context.

Enactor Role: spouse  
Acceptor Role: spouse  

Violation Status: violate  
Violation Status Justification: The conversation involves discussing private marital issues in a public forum, which breaches the expectation of confidentiality.

Violating Action: discussing marital issues publicly  
Violator Role: spouse  
Victim Role: spouse  
Violator Emotion: neutral  
Victim Emotion: neutral
Quality Judgment: inaccurate  
Justification: The conversation does not explicitly reveal private information about a spouse but rather discusses general behaviors and ex

In [79]:
all_data_df['symbolic_annotation'] = all_data_df['id__x'].map(symbols).fillna('')
all_data_df['symbolic_metrics'] = all_data_df['id__x'].map(metrics).fillna({})
all_data_df['symbolic_quality'] = all_data_df['id__x'].map(quality).fillna(False)

In [80]:
all_data_df.to_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_relevance_symbols_data.csv')

In [114]:
qual_data_df = all_data_df[all_data_df['symbolic_quality'] == True]

In [117]:
a, v = 0, 0
for index, row in qual_data_df.iterrows():
    slines = row['symbolic_annotation'].split('\n')
    for sline in slines:
        if 'violation status:' in sline.lower():
            status = sline.lower().strip().split('violation status:')[1]
            if 'violate' in status:
                v += 1
                break
            else:
                a += 1
                break
    else:
        a += 1
print(a, v)     

22811 2891


## Old Code

In [ ]:
prompt_stub = 'You are a helpful assistant. Your task is to judge the relevance of a Chinese cultural social norm to the sitatution in a given conversation.'+\
' Consider factors such as age of the people involved, relationships between them, settings of the conversations such as work, family or friends, topic of conversation and so on.'+\
' Respond with "relevant"/"irrelevant" label. Also provide a justification for your decision. Format your answer as: Justification: <justification>\nRelevance: <decision>'

In [ ]:
gpt_data = []
for eg in data:
    # '\nConversation Summary: ' + eg['summary'].strip() +\
    # '\nNorm Concept: ' + eg['theme'].strip() +\
    prompt = prompt_stub +\
    '\n Conversation in Chinese culture:\n' + eg['dialogue'].strip() + \
    '\nSocial Norm: ' + eg['norm'].strip()
    gpt_data.append((prompt, eg['applicability']))

In [ ]:
print(len(gpt_data))

In [ ]:
from tqdm.auto import tqdm
resps = []
for d in tqdm(gpt_data):
    response = generate_answers_chat([d[0]])
    resps.append(response)

In [ ]:
c, w = 0, 0
X = []
Y = []
idx = 0
for resp, gold in zip(resps, gpt_data):
    g_l = gold[1]
    p_l = None
    for key in resp:
        # print(resp[key], '\n===============\n\n')
        llines = resp[key].strip().split('\n')
        for i in range(len(llines)):
            if llines[len(llines) - 1 - i].strip().startswith('Relevance:'):
                label = llines[len(llines) - 1 - i].strip().split(':')[1].strip().lower()
                break
        p_l = label == 'relevant'
        # print(label)
    if p_l == g_l:
        c += 1
    else:
        w += 1
    X.append(g_l)
    Y.append(p_l)
    # if (not g_l) and g_l != p_l:
    #     print(idx, key, '\n\n')
    #     print(resp[key], '\n-----------------------------------\n')
    idx += 1
print(c, c+w)      

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(X, Y)
print(cm)

In [ ]:
cm[1, 1] / (cm[0, 1] + cm[1, 1])

In [ ]:
X.count(True) / len(X)

In [ ]:
import pickle
with open('./self_verification_4.pkl', 'wb') as outfile:
    pickle.dump((data, gpt_data, resps, X, Y), outfile)

In [ ]:
with open('../distributed_graph_schema/outputs/norm_relevance/final_filtered_criteria.json') as crit_file:
    criteria = json.load(crit_file)

In [ ]:
print(criteria)

In [ ]:
answers = []
for resp in tqdm(resps):
    for key in resp:
        prompt = """Your task is to quantify the output of different tasks based on the given criteria.
            You will be given a criterion a dictionary as follows {"description": criterion description , "accepted_values": possible accepted inputs for this key}.
            You are going to evaluate the test case against the given criterion for the given task.
            Return the assessed performance based on accepted values for each criteria, which must be one of the values provided in the accepted_values list.
            Return only the assessed performance and nothing else""" + \
            "Evaluation dictionary: " + str(criteria) + "\n" + \
            "actual test case to evaluate:\n" + key + '\n' + resp[key].strip()
        response = generate_answers_chat([prompt])
        answers.append(response)
    # break

In [ ]:
c, t = 0, 0
for gold, answer, x, y in zip(gpt_data, answers, X, Y):
    ans_str = str(list(answer.values())[0]).strip().lower()
    quality = None
    for val in criteria['justification quality']['accepted_values']:
        if val in ans_str:
            quality = val
    # print(quality)
    if quality not in ['good', 'fair', 'poor']:
        if y:
            t += 1
            if x:
                c += 1
print(c, t, c / t)

In [ ]:
c, t = 0, 0
for gold, answer, x, y in zip(gpt_data, answers, X, Y):
    ans_str = str(list(answer.values())[0]).strip().lower()
    quality = None
    for val in criteria['justification quality']['accepted_values']:
        if val in ans_str:
            quality = val
    # print(quality)
    if quality not in ['fair', 'poor']:
        if y:
            t += 1
            if x:
                c += 1
print(c, t, c / t)